Part 1 — Connect Google Drive and load your file

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd
import numpy as np
import re

file_path = "/content/drive/MyDrive/IBS_7_Thesis/Python Files/DE_master_file_cleaned.csv"

df_raw = pd.read_csv(file_path, dtype=str, keep_default_na=False)

print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

df_raw.head()

Rows: 104099
Columns: 30


,Name,Rechtsform,Land,PLZ,Ort,HR Amtsgericht,Register-ID,Status,North Data URL,USt.-Id.,...,Eigenkapital EUR,EK-Quote %,EK-Rendite %,Mitarbeiterzahl,Umsatz pro Mitarbeiter EUR,Steuern EUR,Kassenbestand EUR,Forderungen EUR,Verbindlichkeiten EUR,Personalaufwand EUR
0,Knut Rösch Baugrunduntersuchungen GmbH,GmbH,DE,22844.0,Norderstedt,Kiel,HRB 7129 KI,aktiv,https://www.northdata.de/Knut%20R%C3%B6sch%20B...,DE118666165,...,"200.063,46","29,43","-56,73",,,,"150.845,95","59.624,39","358.115,38",
1,Soulworks Developments GmbH,GmbH,DE,83646.0,Bad Tölz,München,HRB 288800,aktiv,https://www.northdata.de/Soulworks%20Developme...,DE260731679,...,"619.387,88","91,14","-35,09",,,,"333.451,04","331.284,83","29.415,29",
2,BEHR INGENIEURE GmbH,GmbH,DE,4103.0,Leipzig,Leipzig,HRB 11586,aktiv,https://www.northdata.de/BEHR%20INGENIEURE%20G...,DE177768304,...,"156.773,48","23,07","85,04",12.0,"91.666,67",,"425.336,85","181.877,25","162.438,7",
3,GLASFAKTOR Ingenieure GmbH,GmbH,DE,1324.0,Dresden,Dresden,HRB 30711,aktiv,https://www.northdata.de/GLASFAKTOR%20Ingenieu...,DE280122771,...,"501.018,27","73,73","43,11",,,,"619.318,74","49.954,88","136.972,16",
4,Novum Analytik GmbH,GmbH,DE,74078.0,Heilbronn,Stuttgart,HRB 723862,aktiv,https://www.northdata.de/Novum%20Analytik%20Gm...,DE255820468,...,"534.165,84","78,62","1,22",13.0,"246.153,85",,"162.635,85","191.789,74","112.454,35",


Part 2 — Making a working Copy

In [6]:
df = df_raw.copy()

print(df.shape)
df.head()

(104099, 30)


,Name,Rechtsform,Land,PLZ,Ort,HR Amtsgericht,Register-ID,Status,North Data URL,USt.-Id.,...,Eigenkapital EUR,EK-Quote %,EK-Rendite %,Mitarbeiterzahl,Umsatz pro Mitarbeiter EUR,Steuern EUR,Kassenbestand EUR,Forderungen EUR,Verbindlichkeiten EUR,Personalaufwand EUR
0,Knut Rösch Baugrunduntersuchungen GmbH,GmbH,DE,22844.0,Norderstedt,Kiel,HRB 7129 KI,aktiv,https://www.northdata.de/Knut%20R%C3%B6sch%20B...,DE118666165,...,"200.063,46","29,43","-56,73",,,,"150.845,95","59.624,39","358.115,38",
1,Soulworks Developments GmbH,GmbH,DE,83646.0,Bad Tölz,München,HRB 288800,aktiv,https://www.northdata.de/Soulworks%20Developme...,DE260731679,...,"619.387,88","91,14","-35,09",,,,"333.451,04","331.284,83","29.415,29",
2,BEHR INGENIEURE GmbH,GmbH,DE,4103.0,Leipzig,Leipzig,HRB 11586,aktiv,https://www.northdata.de/BEHR%20INGENIEURE%20G...,DE177768304,...,"156.773,48","23,07","85,04",12.0,"91.666,67",,"425.336,85","181.877,25","162.438,7",
3,GLASFAKTOR Ingenieure GmbH,GmbH,DE,1324.0,Dresden,Dresden,HRB 30711,aktiv,https://www.northdata.de/GLASFAKTOR%20Ingenieu...,DE280122771,...,"501.018,27","73,73","43,11",,,,"619.318,74","49.954,88","136.972,16",
4,Novum Analytik GmbH,GmbH,DE,74078.0,Heilbronn,Stuttgart,HRB 723862,aktiv,https://www.northdata.de/Novum%20Analytik%20Gm...,DE255820468,...,"534.165,84","78,62","1,22",13.0,"246.153,85",,"162.635,85","191.789,74","112.454,35",


Part 3 - Checking all the Column

In [7]:
for i, col in enumerate(df.columns, start=1):
    print(i, col)

1 Name
2 Rechtsform
3 Land
4 PLZ
5 Ort
6 HR Amtsgericht
7 Register-ID
8 Status
9 North Data URL
10 USt.-Id.
11 Branche (WZ)
12 Gegenstand
13 Finanzkennzahlen Datum
14 Stamm-/Grundkapital EUR
15 Bilanzsumme EUR
16 Gewinn EUR
17 Gewinn CAGR %
18 Umsatz EUR
19 Umsatz CAGR %
20 Umsatzrendite %
21 Eigenkapital EUR
22 EK-Quote %
23 EK-Rendite %
24 Mitarbeiterzahl
25 Umsatz pro Mitarbeiter EUR
26 Steuern EUR
27 Kassenbestand EUR
28 Forderungen EUR
29 Verbindlichkeiten EUR
30 Personalaufwand EUR


Part 4 - Quick Column Overview

In [8]:
column_summary = []

for col in df.columns:
    non_empty = (df[col].astype(str).str.strip() != "").sum()
    empty = (df[col].astype(str).str.strip() == "").sum()
    unique_values = df[col].nunique()

    examples = (
        df.loc[df[col].astype(str).str.strip() != "", col]
        .drop_duplicates()
        .head(5)
        .tolist()
    )

    column_summary.append({
        "column": col,
        "non_empty_rows": non_empty,
        "empty_rows": empty,
        "unique_values": unique_values,
        "example_values": examples
    })

column_summary_df = pd.DataFrame(column_summary)
column_summary_df

,column,non_empty_rows,empty_rows,unique_values,example_values
0,Name,104099,0,95604,"[Knut Rösch Baugrunduntersuchungen GmbH, Soulw..."
1,Rechtsform,104075,24,57,"[GmbH, GmbH & Co. KG, UG, UG & Co. KG, Ltd. & ..."
2,Land,104099,0,5,"[DE, IT, CH, AT, BE]"
3,PLZ,104026,73,7295,"[22844.0, 83646.0, 4103.0, 1324.0, 74078.0]"
4,Ort,104099,0,6916,"[Norderstedt, Bad Tölz, Leipzig, Dresden, Heil..."
5,HR Amtsgericht,104046,53,117,"[Kiel, München, Leipzig, Dresden, Stuttgart]"
6,Register-ID,104086,13,77569,"[HRB 7129 KI, HRB 288800, HRB 11586, HRB 30711..."
7,Status,104099,0,5,"[aktiv, active, liquidation, in Liquidation, e..."
8,North Data URL,104099,0,96227,[https://www.northdata.de/Knut%20R%C3%B6sch%20...
9,USt.-Id.,50638,53461,46194,"[DE118666165, DE260731679, DE177768304, DE2801..."


Part 5 - Define Numeric Column to Clean

In [9]:
numeric_cols = [
    "Stamm-/Grundkapital EUR",
    "Bilanzsumme EUR",
    "Gewinn EUR",
    "Gewinn CAGR %",
    "Umsatz EUR",
    "Umsatz CAGR %",
    "Umsatzrendite %",
    "Eigenkapital EUR",
    "EK-Quote %",
    "EK-Rendite %",
    "Mitarbeiterzahl",
    "Umsatz pro Mitarbeiter EUR",
    "Steuern EUR",
    "Kassenbestand EUR",
    "Forderungen EUR",
    "Verbindlichkeiten EUR",
    "Personalaufwand EUR"
]

numeric_cols

['Stamm-/Grundkapital EUR',
 'Bilanzsumme EUR',
 'Gewinn EUR',
 'Gewinn CAGR %',
 'Umsatz EUR',
 'Umsatz CAGR %',
 'Umsatzrendite %',
 'Eigenkapital EUR',
 'EK-Quote %',
 'EK-Rendite %',
 'Mitarbeiterzahl',
 'Umsatz pro Mitarbeiter EUR',
 'Steuern EUR',
 'Kassenbestand EUR',
 'Forderungen EUR',
 'Verbindlichkeiten EUR',
 'Personalaufwand EUR']

In [54]:
numeric_num_cols = [col + "_num" for col in numeric_cols]

descriptive_summary = df[numeric_num_cols].describe().T

descriptive_summary

,count,mean,std,min,25%,50%,75%,max
Stamm-/Grundkapital EUR_num,98676.0,2.228907e+07,5.704005e+09,1.000000e+00,25000.000,25100.000,5.030000e+04,1.771093e+12
Bilanzsumme EUR_num,98861.0,1.196656e+08,1.097818e+10,1.300000e-01,489968.610,1231556.470,4.523259e+06,2.913900e+12
Gewinn EUR_num,84579.0,1.371382e+05,3.427518e+08,-9.324879e+10,0.000,36210.220,2.154636e+05,1.103883e+10
Gewinn CAGR %_num,42981.0,8.604260e+01,4.525784e+02,-1.000000e+02,-9.830,12.750,5.008000e+01,9.936200e+03
Umsatz EUR_num,98852.0,8.413438e+07,2.769594e+09,-5.751900e+04,1200000.000,3300000.000,9.400000e+06,4.715487e+11
Umsatz CAGR %_num,94990.0,3.661237e+01,2.828597e+02,-1.000000e+02,-1.340,5.200,1.691000e+01,9.909630e+03
Umsatzrendite %_num,84363.0,1.372541e+01,2.224129e+02,-7.754750e+03,0.000,1.000,4.500000e+00,9.801510e+03
Eigenkapital EUR_num,98024.0,5.991846e+08,1.834244e+11,-4.017151e+11,79071.520,395298.440,1.398496e+06,5.742590e+13
EK-Quote %_num,97982.0,4.320279e+01,8.792363e+01,-8.021930e+03,13.560,43.685,7.337000e+01,9.474500e+03
EK-Rendite %_num,76660.0,1.804008e+01,3.230369e+02,-9.872920e+03,0.000,8.895,2.695000e+01,9.936500e+03


In [55]:
descriptive_summary = df[numeric_num_cols].agg([
    "count",
    "mean",
    "std",
    "min",
    "median",
    "max"
]).T

descriptive_summary["missing"] = df[numeric_num_cols].isna().sum()
descriptive_summary["available_%"] = (df[numeric_num_cols].notna().sum() / len(df) * 100).round(2)

descriptive_summary = descriptive_summary[
    ["count", "missing", "available_%", "mean", "std", "min", "median", "max"]
]

descriptive_summary

,count,missing,available_%,mean,std,min,median,max
Stamm-/Grundkapital EUR_num,98676.0,5423,94.79,2.228907e+07,5.704005e+09,1.000000e+00,25100.000,1.771093e+12
Bilanzsumme EUR_num,98861.0,5238,94.97,1.196656e+08,1.097818e+10,1.300000e-01,1231556.470,2.913900e+12
Gewinn EUR_num,84579.0,19520,81.25,1.371382e+05,3.427518e+08,-9.324879e+10,36210.220,1.103883e+10
Gewinn CAGR %_num,42981.0,61118,41.29,8.604260e+01,4.525784e+02,-1.000000e+02,12.750,9.936200e+03
Umsatz EUR_num,98852.0,5247,94.96,8.413438e+07,2.769594e+09,-5.751900e+04,3300000.000,4.715487e+11
Umsatz CAGR %_num,94990.0,9109,91.25,3.661237e+01,2.828597e+02,-1.000000e+02,5.200,9.909630e+03
Umsatzrendite %_num,84363.0,19736,81.04,1.372541e+01,2.224129e+02,-7.754750e+03,1.000,9.801510e+03
Eigenkapital EUR_num,98024.0,6075,94.16,5.991846e+08,1.834244e+11,-4.017151e+11,395298.440,5.742590e+13
EK-Quote %_num,97982.0,6117,94.12,4.320279e+01,8.792363e+01,-8.021930e+03,43.685,9.474500e+03
EK-Rendite %_num,76660.0,27439,73.64,1.804008e+01,3.230369e+02,-9.872920e+03,8.895,9.936500e+03


Part 6 — German number cleaning function

In [10]:
missing_tokens = ["", "nan", "none", "null", "na", "n/a", "-", "—"]

def clean_german_number(series, old_currency_as_missing=True):
    raw = series.astype(str).str.strip()
    raw = raw.str.replace("\u00a0", " ", regex=False)
    raw = raw.str.replace("−", "-", regex=False)

    raw_lower = raw.str.lower()
    is_missing = raw_lower.isin(missing_tokens)

    uncertain_flag = raw.str.contains(r"\?", regex=True, na=False)

    dollar_flag = raw.str.contains(r"\$", regex=True, na=False)
    dem_flag = raw.str.contains(r"\bDEM\b", case=False, regex=True, na=False)
    ats_flag = raw.str.contains(r"\bATS\b", case=False, regex=True, na=False)

    mio_flag = raw.str.contains(r"\bMio\.?", case=False, regex=True, na=False)
    mrd_flag = raw.str.contains(r"\bMrd\.?", case=False, regex=True, na=False)
    tsd_flag = raw.str.contains(r"\bTsd\.?", case=False, regex=True, na=False)

    multiplier = pd.Series(1.0, index=series.index)
    multiplier.loc[tsd_flag] = 1_000
    multiplier.loc[mio_flag] = 1_000_000
    multiplier.loc[mrd_flag] = 1_000_000_000

    cleaned = raw.copy()

    cleaned = cleaned.str.replace(r"[€$%?]", "", regex=True)

    cleaned = cleaned.str.replace(
        r"\bEUR\b|\bDEM\b|\bATS\b|\bMio\.?|\bMrd\.?|\bTsd\.?",
        "",
        case=False,
        regex=True
    )

    cleaned = cleaned.str.replace(" ", "", regex=False)
    cleaned = cleaned.str.replace("'", "", regex=False)

    both_sep = cleaned.str.contains(".", regex=False) & cleaned.str.contains(",", regex=False)
    comma_only = cleaned.str.contains(",", regex=False) & ~cleaned.str.contains(".", regex=False)
    dot_only = cleaned.str.contains(".", regex=False) & ~cleaned.str.contains(",", regex=False)

    cleaned.loc[both_sep] = (
        cleaned.loc[both_sep]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    cleaned.loc[comma_only] = cleaned.loc[comma_only].str.replace(",", ".", regex=False)

    dot_thousand_pattern = r"^[+-]?\d{1,3}(\.\d{3})+$"
    dot_thousand = dot_only & cleaned.str.match(dot_thousand_pattern, na=False)

    cleaned.loc[dot_thousand] = cleaned.loc[dot_thousand].str.replace(".", "", regex=False)

    numeric = pd.to_numeric(cleaned, errors="coerce")
    numeric = numeric * multiplier

    numeric.loc[is_missing] = np.nan

    if old_currency_as_missing:
        numeric.loc[dem_flag | ats_flag] = np.nan

    numeric.loc[dollar_flag] = np.nan

    problem_flag = uncertain_flag | dollar_flag | dem_flag | ats_flag

    return numeric, problem_flag

Part 7 — Apply cleaning to all numeric columns


In [11]:
for col in numeric_cols:
    df[col + "_num"], df[col + "_problem_flag"] = clean_german_number(df[col])

print("Numeric cleaning completed.")

Numeric cleaning completed.


Part 8 — Check before and after conversion

In [12]:
check_cols = [
    "Bilanzsumme EUR",
    "Gewinn EUR",
    "Umsatz EUR",
    "Eigenkapital EUR",
    "Verbindlichkeiten EUR",
    "Kassenbestand EUR",
    "Forderungen EUR",
    "Mitarbeiterzahl",
    "Stamm-/Grundkapital EUR"
]

for col in check_cols:
    print("\nCOLUMN:", col)
    display(df[[col, col + "_num", col + "_problem_flag"]].head(10))


COLUMN: Bilanzsumme EUR


,Bilanzsumme EUR,Bilanzsumme EUR_num,Bilanzsumme EUR_problem_flag
0,"679.793,84",679793.84,False
1,"679.593,17",679593.17,False
2,"679.558,8",679558.80,False
3,"679.529,43",679529.43,False
4,"679.420,19",679420.19,False
5,"679.313,87",679313.87,False
6,"679.249,22",679249.22,False
7,"679.168,75",679168.75,False
8,"679.101,39",679101.39,False
9,"679.085,83",679085.83,False



COLUMN: Gewinn EUR


,Gewinn EUR,Gewinn EUR_num,Gewinn EUR_problem_flag
0,"-113.496,26",-113496.26,False
1,"-217.337,80",-217337.80,False
2,"133.323,1",133323.10,False
3,"216.008,68",216008.68,False
4,"6.529,39",6529.39,False
5,0,0.00,False
6,"111.731,97",111731.97,False
7,"225.663,59",225663.59,False
8,"82.210,60",82210.60,False
9,"82.402,34",82402.34,False



COLUMN: Umsatz EUR


...

,Forderungen EUR,Forderungen EUR_num,Forderungen EUR_problem_flag
0,"59.624,39",59624.39,False
1,"331.284,83",331284.83,False
2,"181.877,25",181877.25,False
3,"49.954,88",49954.88,False
4,"191.789,74",191789.74,False
5,"1.282,57",1282.57,False
6,"83.703,89",83703.89,False
7,"153.923,52",153923.52,False
8,"102.142,26",102142.26,False
9,"130.898,81",130898.81,False



COLUMN: Mitarbeiterzahl


,Mitarbeiterzahl,Mitarbeiterzahl_num,Mitarbeiterzahl_problem_flag
0,,NaN,False
1,,NaN,False
2,12.0,12.0,False
3,,NaN,False
4,13.0,13.0,False
5,,NaN,False
6,,NaN,False
7,,NaN,False
8,12.0,12.0,False
9,,NaN,False



COLUMN: Stamm-/Grundkapital EUR


,Stamm-/Grundkapital EUR,Stamm-/Grundkapital EUR_num,Stamm-/Grundkapital EUR_problem_flag
0,26.000,26000.0,False
1,25.000,25000.0,False
2,25.600,25600.0,False
3,25.000,25000.0,False
4,25.000,25000.0,False
5,118.000,118000.0,False
6,200.000,200000.0,False
7,25.000,25000.0,False
8,25.000,25000.0,False
9,25.000,25000.0,False


Part 9 — Conversion quality summary

In [13]:
conversion_summary = []

for col in numeric_cols:
    original_non_empty = (df[col].astype(str).str.strip() != "").sum()
    converted_numeric = df[col + "_num"].notna().sum()
    problem_flags = df[col + "_problem_flag"].sum()

    conversion_summary.append({
        "column": col,
        "original_non_empty": original_non_empty,
        "converted_numeric": converted_numeric,
        "not_converted": original_non_empty - converted_numeric,
        "problem_flags": problem_flags,
        "conversion_rate_%": round((converted_numeric / original_non_empty) * 100, 2) if original_non_empty > 0 else 0
    })

conversion_summary_df = pd.DataFrame(conversion_summary)
conversion_summary_df.sort_values("converted_numeric", ascending=False)

,column,original_non_empty,converted_numeric,not_converted,problem_flags,conversion_rate_%
1,Bilanzsumme EUR,98862,98861,1,1,100.00
4,Umsatz EUR,98853,98852,1,2,100.00
0,Stamm-/Grundkapital EUR,99603,98676,927,927,99.07
7,Eigenkapital EUR,98025,98024,1,1632,100.00
8,EK-Quote %,97982,97982,0,1630,100.00
15,Verbindlichkeiten EUR,96669,96668,1,1,100.00
5,Umsatz CAGR %,94990,94990,0,4190,100.00
14,Forderungen EUR,84968,84967,1,1,100.00
13,Kassenbestand EUR,84773,84772,1,1,100.00
2,Gewinn EUR,84580,84579,1,3848,100.00


Part 10 — Clean important non-numeric columns

Part 10.1 - Clean PLZ

In [14]:
df["PLZ_clean"] = (
    df["PLZ"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

df["PLZ_clean"] = df["PLZ_clean"].apply(
    lambda x: x.zfill(5) if x.isdigit() else x
)

df[["PLZ", "PLZ_clean"]].head(10)

,PLZ,PLZ_clean
0,22844.0,22844
1,83646.0,83646
2,4103.0,04103
3,1324.0,01324
4,74078.0,74078
5,95445.0,95445
6,75217.0,75217
7,89522.0,89522
8,96231.0,96231
9,83313.0,83313


Part 10.2 - Clean financial date and Create Year

In [15]:
df["Finanzkennzahlen Datum_clean"] = pd.to_datetime(
    df["Finanzkennzahlen Datum"],
    format="%d.%m.%Y",
    errors="coerce"
)

df["Jahr"] = df["Finanzkennzahlen Datum_clean"].dt.year

df[["Finanzkennzahlen Datum", "Finanzkennzahlen Datum_clean", "Jahr"]].head(10)

,Finanzkennzahlen Datum,Finanzkennzahlen Datum_clean,Jahr
0,31.12.2024,2024-12-31,2024.0
1,31.12.2022,2022-12-31,2022.0
2,31.12.2023,2023-12-31,2023.0
3,30.11.2023,2023-11-30,2023.0
4,31.12.2023,2023-12-31,2023.0
5,31.12.2022,2022-12-31,2022.0
6,30.09.2024,2024-09-30,2024.0
7,31.12.2023,2023-12-31,2023.0
8,31.12.2024,2024-12-31,2024.0
9,31.12.2021,2021-12-31,2021.0


In [16]:
df["Jahr"].value_counts(dropna=False).sort_index()

,count
Jahr,
2006.0,5
2007.0,4
2008.0,4
2009.0,11
2010.0,27
2011.0,162
2012.0,95
2013.0,40
2014.0,53


Part 10.3 - Cleaning Company Status

In [17]:
def clean_status(x):
    x = str(x).strip().lower()

    if x in ["aktiv", "active"]:
        return "active"

    elif "liquidation" in x or "liquidierung" in x:
        return "liquidation"

    elif "erloschen" in x or "gelöscht" in x or "inactive" in x:
        return "inactive"

    elif x == "":
        return np.nan

    else:
        return x

df["Status_clean"] = df["Status"].apply(clean_status)

df["Status_clean"].value_counts(dropna=False)

,count
Status_clean,
active,99879
inactive,2335
liquidation,1885


Part 10.4 - Spliting Industry Columns - Branche(WZ)

In [18]:
df["WZ_Code"] = df["Branche (WZ)"].str.extract(r"^(\d{2}(?:\.\d{1,2})*)")

df["WZ_Division"] = df["WZ_Code"].str.extract(r"^(\d{2})")

df["Branche_Text"] = df["Branche (WZ)"].str.replace(
    r"^\d{2}(?:\.\d{1,2})*\s*",
    "",
    regex=True
)

df[["Branche (WZ)", "WZ_Code", "WZ_Division", "Branche_Text"]].head(10)

,Branche (WZ),WZ_Code,WZ_Division,Branche_Text
0,71.12.9 Sonstige Ingenieurbüros,71.12.9,71,Sonstige Ingenieurbüros
1,71.11.1 Architekturbüros für Hochbau,71.11.1,71,Architekturbüros für Hochbau
2,71.12.1 Ingenieurbüros für bautechnische Gesam...,71.12.1,71,Ingenieurbüros für bautechnische Gesamtplanung
3,71.12.1 Ingenieurbüros für bautechnische Gesam...,71.12.1,71,Ingenieurbüros für bautechnische Gesamtplanung
4,"71.20.0 Technische, physikalische und chemisch...",71.20.0,71,"Technische, physikalische und chemische Unters..."
5,71.12.2 Ingenieurbüros für technische Fachplan...,71.12.2,71,Ingenieurbüros für technische Fachplanung und ...
6,71.12.2 Ingenieurbüros für technische Fachplan...,71.12.2,71,Ingenieurbüros für technische Fachplanung und ...
7,"71.20.0 Technische, physikalische und chemisch...",71.20.0,71,"Technische, physikalische und chemische Unters..."
8,71.12.3 Vermessungsbüros,71.12.3,71,Vermessungsbüros
9,71.11.1 Architekturbüros für Hochbau,71.11.1,71,Architekturbüros für Hochbau


## Part 11 — Create detailed problem flags

Part 11.1 — Create separate flags

In [19]:
for col in numeric_cols:
    raw = df[col].astype(str)

    df[col + "_flag_uncertain"] = raw.str.contains(r"\?", regex=True, na=False)
    df[col + "_flag_dollar"] = raw.str.contains(r"\$", regex=True, na=False)
    df[col + "_flag_DEM"] = raw.str.contains(r"\bDEM\b", case=False, regex=True, na=False)
    df[col + "_flag_ATS"] = raw.str.contains(r"\bATS\b", case=False, regex=True, na=False)
    df[col + "_flag_mio"] = raw.str.contains(r"\bMio\.?", case=False, regex=True, na=False)
    df[col + "_flag_mrd"] = raw.str.contains(r"\bMrd\.?", case=False, regex=True, na=False)

print("Detailed problem flags created.")

/tmp/ipykernel_416/3186382060.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_flag_mio"] = raw.str.contains(r"\bMio\.?", case=False, regex=True, na=False)
/tmp/ipykernel_416/3186382060.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_flag_mrd"] = raw.str.contains(r"\bMrd\.?", case=False, regex=True, na=False)
/tmp/ipykernel_416/3186382060.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider jo

Detailed problem flags created.


/tmp/ipykernel_416/3186382060.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_flag_mio"] = raw.str.contains(r"\bMio\.?", case=False, regex=True, na=False)
/tmp/ipykernel_416/3186382060.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_flag_mrd"] = raw.str.contains(r"\bMrd\.?", case=False, regex=True, na=False)
/tmp/ipykernel_416/3186382060.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider jo

Part 11.2 — Create better issue summary table

In [20]:
issue_summary = []

for col in numeric_cols:
    original_non_empty = (df[col].astype(str).str.strip() != "").sum()
    converted_numeric = df[col + "_num"].notna().sum()

    issue_summary.append({
        "column": col,
        "original_non_empty": original_non_empty,
        "converted_numeric": converted_numeric,
        "not_converted": original_non_empty - converted_numeric,
        "uncertain_question_mark": df[col + "_flag_uncertain"].sum(),
        "dollar_values": df[col + "_flag_dollar"].sum(),
        "DEM_values": df[col + "_flag_DEM"].sum(),
        "ATS_values": df[col + "_flag_ATS"].sum(),
        "Mio_values": df[col + "_flag_mio"].sum(),
        "Mrd_values": df[col + "_flag_mrd"].sum(),
        "conversion_rate_%": round((converted_numeric / original_non_empty) * 100, 4)
        if original_non_empty > 0 else 0
    })

issue_summary_df = pd.DataFrame(issue_summary)

issue_summary_df.sort_values("not_converted", ascending=False)

,column,original_non_empty,converted_numeric,not_converted,uncertain_question_mark,dollar_values,DEM_values,ATS_values,Mio_values,Mrd_values,conversion_rate_%
0,Stamm-/Grundkapital EUR,99603,98676,927,0,0,926,1,81,2,99.0693
1,Bilanzsumme EUR,98862,98861,1,0,1,0,0,2392,0,99.9990
2,Gewinn EUR,84580,84579,1,3847,1,0,0,149,0,99.9988
4,Umsatz EUR,98853,98852,1,1,1,0,0,3086,4,99.9990
7,Eigenkapital EUR,98025,98024,1,1631,1,0,0,1264,2,99.9990
13,Kassenbestand EUR,84773,84772,1,0,1,0,0,163,1,99.9988
14,Forderungen EUR,84968,84967,1,0,1,0,0,444,1,99.9988
11,Umsatz pro Mitarbeiter EUR,32749,32748,1,0,1,0,0,111,0,99.9969
12,Steuern EUR,1326,1325,1,10,1,0,0,8,0,99.9246
15,Verbindlichkeiten EUR,96669,96668,1,0,1,0,0,960,3,99.9990


Part 11.3 — Show exact values that were not converted

In [21]:
for col in numeric_cols:
    mask = (
        (df[col].astype(str).str.strip() != "") &
        (df[col + "_num"].isna())
    )

    if mask.sum() > 0:
        print("\n==============================")
        print("COLUMN:", col)
        print("Not converted rows:", mask.sum())
        print("==============================")

        display(
            df.loc[
                mask,
                ["Name", "Register-ID", "Finanzkennzahlen Datum", col]
            ].head(20)
        )


COLUMN: Stamm-/Grundkapital EUR
Not converted rows: 927


,Name,Register-ID,Finanzkennzahlen Datum,Stamm-/Grundkapital EUR
5420,CURA Pflegehaus Seligenstadt GmbH,HRB 33667,31.12.2023,50.000 DEM
5424,Dr. Grossmann Wirtschaftsberatung GmbH,HRB 49622,31.12.2023,100.000 DEM
5426,Daniel Binder Vermögensverwaltung GmbH,HRB 541640,31.12.2023,50.000 DEM
5452,Raulino Vermögensverwaltung GmbH,HRB 106961,31.12.2022,50.000 DEM
5462,MEBA Vermögensverwaltung GmbH,HRB 93511,31.12.2022,50.000 DEM
5477,Stollar GmbH,HRB 37011,31.12.2021,100.000 DEM
5485,Werner Kappel Vermögensverwaltungs GmbH,HRB 3105,31.12.2021,50.000 DEM
5492,IFV Gesellschaft für Finanz- und Vermögensplan...,HRB 78479,31.12.2022,50.000 DEM
5519,Vierte Dreiländer Handels- und Beteiligungsges...,HRA 12109,31.12.2024,18 Mio. DEM
5554,Rainer Heberer Verwaltungs GmbH,HRB 8791,31.12.2020,100.000 DEM



COLUMN: Bilanzsumme EUR
Not converted rows: 1


,Name,Register-ID,Finanzkennzahlen Datum,Bilanzsumme EUR
59384,Neuberger Berman Asset Management Ireland Ltd....,HRB 114969,31.12.2024,"33,05 $"



COLUMN: Gewinn EUR
Not converted rows: 1


...

,Name,Register-ID,Finanzkennzahlen Datum,Forderungen EUR
59384,Neuberger Berman Asset Management Ireland Ltd....,HRB 114969,31.12.2024,"33,44 $"



COLUMN: Verbindlichkeiten EUR
Not converted rows: 1


,Name,Register-ID,Finanzkennzahlen Datum,Verbindlichkeiten EUR
59384,Neuberger Berman Asset Management Ireland Ltd....,HRB 114969,31.12.2024,"55,16 $"



COLUMN: Personalaufwand EUR
Not converted rows: 1


,Name,Register-ID,Finanzkennzahlen Datum,Personalaufwand EUR
59384,Neuberger Berman Asset Management Ireland Ltd....,HRB 114969,31.12.2024,"33,74 $"


Part 11.4 — Show unique not-converted values

In [22]:
for col in numeric_cols:
    mask = (
        (df[col].astype(str).str.strip() != "") &
        (df[col + "_num"].isna())
    )

    if mask.sum() > 0:
        print("\nCOLUMN:", col)
        print(df.loc[mask, col].value_counts().head(20))


COLUMN: Stamm-/Grundkapital EUR
Stamm-/Grundkapital EUR
50.000 DEM       605
100.000 DEM       99
200.000 DEM       27
500.000 DEM       20
51.000 DEM        20
1.000.000 DEM     12
150.000 DEM       12
300.000 DEM       12
60.000 DEM        10
250.000 DEM       10
400.000 DEM        7
80.000 DEM         6
10.000,00 DEM      4
1.500.000 DEM      4
75.000 DEM         4
10 Mio. DEM        3
600.000 DEM        3
90.000 DEM         3
120.000 DEM        3
800.000 DEM        3
Name: count, dtype: int64

COLUMN: Bilanzsumme EUR
Bilanzsumme EUR
33,05 $    1
Name: count, dtype: int64

COLUMN: Gewinn EUR
Gewinn EUR
7,77 $    1
Name: count, dtype: int64

COLUMN: Umsatz EUR
Umsatz EUR
257,33 $    1
Name: count, dtype: int64

COLUMN: Eigenkapital EUR
Eigenkapital EUR
33,05 $    1
Name: count, dtype: int64

COLUMN: Umsatz pro Mitarbeiter EUR
Umsatz pro Mitarbeiter EUR
2,80 $    1
Name: count, dtype: int64

COLUMN: Steuern EUR
Steuern EUR
1,73 $    1
Name: count, dtype: int64

COLUMN: Kassenbestand 

## Part 12 - Creating Ratios

Part 12.1 - Safe Ratio Construction


In [34]:
def safe_ratio(numerator, denominator, min_denominator=1):
    return np.where(
        (numerator.notna()) &
        (denominator.notna()) &
        (denominator >= min_denominator),
        numerator / denominator,
        np.nan
    )

12.2 - Create main Financial Ratio

In [36]:
df["profit_margin"] = safe_ratio(
    df["Gewinn EUR_num"],
    df["Umsatz EUR_num"],
    min_denominator=1000
)

df["roa"] = safe_ratio(
    df["Gewinn EUR_num"],
    df["Bilanzsumme EUR_num"],
    min_denominator=1000
)

df["roe"] = safe_ratio(
    df["Gewinn EUR_num"],
    df["Eigenkapital EUR_num"],
    min_denominator=1000
)

df["equity_ratio_calc"] = safe_ratio(
    df["Eigenkapital EUR_num"],
    df["Bilanzsumme EUR_num"],
    min_denominator=1000
)

df["debt_ratio"] = safe_ratio(
    df["Verbindlichkeiten EUR_num"],
    df["Bilanzsumme EUR_num"],
    min_denominator=1000
)

df["cash_to_assets"] = safe_ratio(
    df["Kassenbestand EUR_num"],
    df["Bilanzsumme EUR_num"],
    min_denominator=1000
)

df["receivables_to_assets"] = safe_ratio(
    df["Forderungen EUR_num"],
    df["Bilanzsumme EUR_num"],
    min_denominator=1000
)

df["revenue_per_employee_calc"] = safe_ratio(
    df["Umsatz EUR_num"],
    df["Mitarbeiterzahl_num"]
)

df["profit_per_employee"] = safe_ratio(
    df["Gewinn EUR_num"],
    df["Mitarbeiterzahl_num"]
)

df["asset_turnover"] = safe_ratio(
    df["Umsatz EUR_num"],
    df["Bilanzsumme EUR_num"],
    min_denominator=1000
)

# 1. Debt-to-Equity Ratio
# Measures financial leverage (Total Liabilities / Total Equity)
df["debt_to_equity_ratio"] = safe_ratio(
    df["Verbindlichkeiten EUR_num"],
    df["Eigenkapital EUR_num"],
    min_denominator=1000
)

# 2. Cash Ratio
# Measures immediate liquidity (Cash / Total Liabilities)
df["cash_ratio"] = safe_ratio(
    df["Kassenbestand EUR_num"],
    df["Verbindlichkeiten EUR_num"],
    min_denominator=1000
)

# 3. Receivables Turnover
# Measures efficiency in collecting debt (Revenue / Accounts Receivable)
df["receivables_turnover"] = safe_ratio(
    df["Umsatz EUR_num"],
    df["Forderungen EUR_num"],
    min_denominator=1000
)

print("Ratios created successfully.")

Ratios created successfully.


In [45]:
ratio_cols = [
    "profit_margin",
    "roa",
    "roe",
    "equity_ratio_calc",
    "debt_ratio",
    "cash_to_assets",
    "receivables_to_assets",
    "revenue_per_employee_calc",
    "profit_per_employee",
    "asset_turnover",
    "debt_to_equity_ratio",
    "cash_ratio",
    "receivables_turnover"
]

def winsorize_series(series, lower=0.01, upper=0.99):
    lower_bound = series.quantile(lower)
    upper_bound = series.quantile(upper)
    return series.clip(lower_bound, upper_bound)

for col in ratio_cols:
    df[col + "_winsor"] = winsorize_series(df[col])

/tmp/ipykernel_416/4051630669.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_winsor"] = winsorize_series(df[col])
/tmp/ipykernel_416/4051630669.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_winsor"] = winsorize_series(df[col])


12.3 - Checking availibility of Ratios

In [46]:
ratio_cols = [
    "profit_margin",
    "roa",
    "roe",
    "equity_ratio_calc",
    "debt_ratio",
    "cash_to_assets",
    "receivables_to_assets",
    "revenue_per_employee_calc",
    "profit_per_employee",
    "asset_turnover",
    "debt_to_equity_ratio",
    "cash_ratio",
    "receivables_turnover"
]

ratio_summary = []

for col in ratio_cols:
    usable = df[col].notna().sum()

    ratio_summary.append({
        "ratio": col,
        "usable_rows": usable,
        "missing_rows": df[col].isna().sum(),
        "usable_%": round((usable / len(df)) * 100, 2)
    })

ratio_summary_df = pd.DataFrame(ratio_summary)

ratio_summary_df.sort_values("usable_rows", ascending=False)

,ratio,usable_rows,missing_rows,usable_%
9,asset_turnover,98635,5464,94.75
3,equity_ratio_calc,97811,6288,93.96
4,debt_ratio,96524,7575,92.72
10,debt_to_equity_ratio,87787,16312,84.33
6,receivables_to_assets,84893,19206,81.55
5,cash_to_assets,84662,19437,81.33
1,roa,84436,19663,81.11
0,profit_margin,84302,19797,80.98
12,receivables_turnover,82390,21709,79.15
11,cash_ratio,81675,22424,78.46


Part 13 — Create filtered analysis dataset

In [47]:
analysis_df = df[
    (df["Land"].str.upper() == "DE") &
    (df["Status_clean"] == "active") &
    (df["Jahr"].notna())
].copy()

print("Original rows:", len(df))
print("Rows after filtering:", len(analysis_df))

analysis_df.head()

Original rows: 104099
Rows after filtering: 96691


,Name,Rechtsform,Land,PLZ,Ort,HR Amtsgericht,Register-ID,Status,North Data URL,USt.-Id.,...,equity_ratio_calc_winsor,debt_ratio_winsor,cash_to_assets_winsor,receivables_to_assets_winsor,asset_turnover_winsor,debt_to_equity_ratio_winsor,cash_ratio_winsor,receivables_turnover_winsor,revenue_per_employee_calc_winsor,profit_per_employee_winsor
0,Knut Rösch Baugrunduntersuchungen GmbH,GmbH,DE,22844.0,Norderstedt,Kiel,HRB 7129 KI,aktiv,https://www.northdata.de/Knut%20R%C3%B6sch%20B...,DE118666165,...,0.294300,0.526800,0.221900,0.087710,1.015014,1.790009,0.421222,11.572445,NaN,NaN
1,Soulworks Developments GmbH,GmbH,DE,83646.0,Bad Tölz,München,HRB 288800,aktiv,https://www.northdata.de/Soulworks%20Developme...,DE260731679,...,0.911410,0.043284,0.490663,0.487475,6.327315,0.047491,11.335977,12.979767,NaN,NaN
2,BEHR INGENIEURE GmbH,GmbH,DE,4103.0,Leipzig,Leipzig,HRB 11586,aktiv,https://www.northdata.de/BEHR%20INGENIEURE%20G...,DE177768304,...,0.230699,0.239036,0.625901,0.267640,1.618697,1.036136,2.618445,6.048035,91666.666667,11110.258333
3,GLASFAKTOR Ingenieure GmbH,GmbH,DE,1324.0,Dresden,Dresden,HRB 30711,aktiv,https://www.northdata.de/GLASFAKTOR%20Ingenieu...,DE280122771,...,0.737302,0.201569,0.911394,0.073514,1.618767,0.273388,4.521494,22.019871,NaN,NaN
4,Novum Analytik GmbH,GmbH,DE,74078.0,Heilbronn,Stuttgart,HRB 723862,aktiv,https://www.northdata.de/Novum%20Analytik%20Gm...,DE255820468,...,0.786208,0.165515,0.239374,0.282284,4.709898,0.210523,1.446239,16.684938,246153.846154,502.260769






Part 13.1 — Check what was removed

In [48]:
print("Country distribution:")
display(df["Land"].value_counts(dropna=False).head(20))

print("Status distribution:")
display(df["Status_clean"].value_counts(dropna=False))

print("Rows without year:", df["Jahr"].isna().sum())


Country distribution:


,count
Land,
DE,104093
CH,2
AT,2
IT,1
BE,1


Status distribution:


,count
Status_clean,
active,99879
inactive,2335
liquidation,1885


Rows without year: 5187


## Part 14 — Select only useful thesis columns

In [49]:
final_cols = [
    "Name",
    "Register-ID",
    "Rechtsform",
    "Land",
    "PLZ_clean",
    "Ort",
    "Status_clean",
    "North Data URL",

    "WZ_Code",
    "WZ_Division",
    "Branche_Text",

    "Finanzkennzahlen Datum_clean",
    "Jahr",

    "Stamm-/Grundkapital EUR_num",
    "Bilanzsumme EUR_num",
    "Gewinn EUR_num",
    "Umsatz EUR_num",
    "Eigenkapital EUR_num",
    "Verbindlichkeiten EUR_num",
    "Kassenbestand EUR_num",
    "Forderungen EUR_num",
    "Mitarbeiterzahl_num",

    "profit_margin",
    "roa",
    "roe",
    "equity_ratio_calc",
    "debt_ratio",
    "cash_to_assets",
    "receivables_to_assets",
    "revenue_per_employee_calc",
    "profit_per_employee",
    "asset_turnover",
    "debt_to_equity_ratio",
    "cash_ratio",
    "receivables_turnover"

]

final_analysis_df = analysis_df[final_cols].copy()

print("Final analysis dataset shape:", final_analysis_df.shape)

final_analysis_df.head()

Final analysis dataset shape: (96691, 35)


,Name,Register-ID,Rechtsform,Land,PLZ_clean,Ort,Status_clean,North Data URL,WZ_Code,WZ_Division,...,equity_ratio_calc,debt_ratio,cash_to_assets,receivables_to_assets,revenue_per_employee_calc,profit_per_employee,asset_turnover,debt_to_equity_ratio,cash_ratio,receivables_turnover
0,Knut Rösch Baugrunduntersuchungen GmbH,HRB 7129 KI,GmbH,DE,22844,Norderstedt,active,https://www.northdata.de/Knut%20R%C3%B6sch%20B...,71.12.9,71,...,0.294300,0.526800,0.221900,0.087710,NaN,NaN,1.015014,1.790009,0.421222,11.572445
1,Soulworks Developments GmbH,HRB 288800,GmbH,DE,83646,Bad Tölz,active,https://www.northdata.de/Soulworks%20Developme...,71.11.1,71,...,0.911410,0.043284,0.490663,0.487475,NaN,NaN,6.327315,0.047491,11.335977,12.979767
2,BEHR INGENIEURE GmbH,HRB 11586,GmbH,DE,04103,Leipzig,active,https://www.northdata.de/BEHR%20INGENIEURE%20G...,71.12.1,71,...,0.230699,0.239036,0.625901,0.267640,91666.666667,11110.258333,1.618697,1.036136,2.618445,6.048035
3,GLASFAKTOR Ingenieure GmbH,HRB 30711,GmbH,DE,01324,Dresden,active,https://www.northdata.de/GLASFAKTOR%20Ingenieu...,71.12.1,71,...,0.737302,0.201569,0.911394,0.073514,NaN,NaN,1.618767,0.273388,4.521494,22.019871
4,Novum Analytik GmbH,HRB 723862,GmbH,DE,74078,Heilbronn,active,https://www.northdata.de/Novum%20Analytik%20Gm...,71.20.0,71,...,0.786208,0.165515,0.239374,0.282284,246153.846154,502.260769,4.709898,0.210523,1.446239,16.684938


# **Part 15 — Missing value check after filtering**

In [50]:
missing_final = pd.DataFrame({
    "column": final_analysis_df.columns,
    "available_rows": final_analysis_df.notna().sum().values,
    "missing_rows": final_analysis_df.isna().sum().values,
    "available_%": (final_analysis_df.notna().sum().values / len(final_analysis_df) * 100).round(2)
})

missing_final.sort_values("available_%", ascending=True)

,column,available_rows,missing_rows,available_%
30,profit_per_employee,30340,66351,31.38
29,revenue_per_employee_calc,32250,64441,33.35
21,Mitarbeiterzahl_num,32268,64423,33.37
24,roe,76541,20150,79.16
33,cash_ratio,80208,16483,82.95
34,receivables_turnover,80950,15741,83.72
22,profit_margin,82663,14028,85.49
23,roa,82808,13883,85.64
15,Gewinn EUR_num,82892,13799,85.73
27,cash_to_assets,83010,13681,85.85


# **Part 16 — Basic descriptive statistics**

In [51]:
main_numeric_for_summary = [
    "Bilanzsumme EUR_num",
    "Gewinn EUR_num",
    "Umsatz EUR_num",
    "Eigenkapital EUR_num",
    "Verbindlichkeiten EUR_num",
    "Kassenbestand EUR_num",
    "Forderungen EUR_num",
    "Mitarbeiterzahl_num",
    "profit_margin",
    "roa",
    "roe",
    "equity_ratio_calc",
    "debt_ratio",
    "cash_to_assets",
    "receivables_to_assets",
    "revenue_per_employee_calc",
    "profit_per_employee",
    "asset_turnover",
    "debt_to_equity_ratio",
    "cash_ratio",
    "receivables_turnover"
]

final_analysis_df[main_numeric_for_summary].describe().T

,count,mean,std,min,25%,50%,75%,max
Bilanzsumme EUR_num,96654.0,1.221545e+08,1.110266e+10,1.000000e+00,4.987676e+05,1.255263e+06,4.612110e+06,2.913900e+12
Gewinn EUR_num,82892.0,2.182193e+05,3.454631e+08,-9.324879e+10,0.000000e+00,3.779962e+04,2.202196e+05,1.103883e+10
Umsatz EUR_num,96644.0,8.585696e+07,2.800921e+09,-5.751900e+04,1.200000e+06,3.400000e+06,9.500000e+06,4.715487e+11
Eigenkapital EUR_num,95844.0,6.127124e+08,1.854987e+11,-4.017151e+11,8.553159e+04,4.057052e+05,1.436486e+06,5.742590e+13
Verbindlichkeiten EUR_num,94608.0,8.064836e+07,9.892100e+09,0.000000e+00,8.850230e+04,3.738277e+05,1.454596e+06,2.025742e+12
Kassenbestand EUR_num,83067.0,5.669943e+07,1.533148e+10,-2.209675e+05,3.699975e+04,1.845055e+05,5.726758e+05,4.417900e+12
Forderungen EUR_num,83329.0,9.248021e+07,2.487516e+10,-6.522100e+04,8.415568e+04,2.606070e+05,8.180545e+05,7.180100e+12
Mitarbeiterzahl_num,32268.0,4.785622e+01,5.287607e+02,5.000000e-01,5.000000e+00,1.200000e+01,2.800000e+01,4.600000e+04
profit_margin,82663.0,4.135201e-01,7.359438e+01,-2.089786e+03,0.000000e+00,1.041706e-02,4.542730e-02,2.060371e+04
roa,82808.0,2.929520e-01,5.641586e+01,-4.083709e+02,0.000000e+00,3.105047e-02,1.124586e-01,1.621049e+04
